# Postgres inspection
Look inside the `bluecore` database: tables, row counts, recent rows, and queries running now. Read-only queries only.

Locally it connects with your stack's database login (from the stack checkout's `.env`, else the documented defaults) on `host.docker.internal:5432`. A deployed database is only reachable if you have access (for example an SSH tunnel) and set `BLUECORE_DEV_DB_URL` in `.env.notebooks`.
If your local stack doesn't publish Postgres's port, connect JupyterLab to the stack's network instead, then set `BLUECORE_DB_URL` to use the host `postgres`:

```bash
docker network ls | grep bluecore            # find the stack's network
docker network connect <that network> bluecore-dev-dashboard-jupyter-1
```

In [ ]:
# Your local stack, or dev / stage: pick it on the Notebooks page.
from bluecore import DASHBOARD_API, show, target
bc = target()
bc

In [ ]:
import pandas as pd
conn = bc.db()  # read-only

def q(sql, *params):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])

q("select current_database() as db, current_user as user, version()")

## Tables and their sizes

In [ ]:
q("""
select relname as table, n_live_tup as rows, pg_size_pretty(pg_total_relation_size(relid)) as size
from pg_stat_user_tables
order by n_live_tup desc
""")

## Records by kind
Hubs, works, instances, other resources and profiles all live in `resource_base`.

In [ ]:
q("select type, count(*) as rows, max(updated_at) as last_updated from resource_base group by type order by rows desc")

## Recent rows of one table
Change `TABLE` (from the list above).

In [ ]:
TABLE = "resource_base"
cols = q("select column_name from information_schema.columns where table_name = %s order by ordinal_position", TABLE)
order = next((c for c in ["updated_at", "created_at", "id"] if c in set(cols["column_name"])), None)
skip = {"data_vector", "title_vector"}   # long search vectors
shown = ", ".join(f'"{c}"' for c in cols["column_name"] if c not in skip)
q(f'select {shown} from "{TABLE}"' + (f' order by "{order}" desc' if order else "") + " limit 10")

## Slow or stuck queries right now

In [ ]:
q("""
select pid, state, now() - query_start as running_for, left(query, 120) as query
from pg_stat_activity
where datname = current_database() and state <> 'idle' and pid <> pg_backend_pid()
order by query_start
""")